# Eksamensprojekt: Forudsig travlhed i fitnesscenter (DNN)

Jeg bruger et Deep Neural Network (DNN) med Keras `Sequential` og `Dense` til at forudsige, om et fitnesscenter er roligt, moderat eller travlt.

**Datasæt:** [Crowdedness at the Campus Gym (Kaggle)](https://www.kaggle.com/datasets/nsrose7224/crowdedness-at-the-campus-gym)

## 1. Problem

Jeg vil forudsige, hvornår et fitnesscenter er travlt. Jeg træner en DNN til at klassificere en måling som:

- **0 = Roligt** (≤ 20 personer)
- **1 = Moderat** (21–50 personer)
- **2 = Travlt** (> 50 personer)

Jeg bruger features som time, ugedag, temperatur og semester-info.

## 2. Import af biblioteker

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers
from tensorflow.keras.utils import to_categorical

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 3. Indlæs datasæt

Jeg indlæser `data.csv`. Hvis filen mangler, downloader jeg den automatisk.

In [ ]:
from pathlib import Path
import urllib.request

candidates = [
    Path("data.csv"),
    Path("../data/data.csv"),
    Path("data/data.csv"),
    Path("/content/data.csv"),
]

data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    data_path = Path("data.csv")
    url = (
        "https://raw.githubusercontent.com/hannahli2014/"
        "Gym-Crowdedness-Analysis/master/data.csv"
    )
    print(f"Downloader data.csv fra:\n{url}")
    urllib.request.urlretrieve(url, data_path)

df = pd.read_csv(data_path)
print(f"Indlæst: {data_path.resolve()}")
print(f"Shape: {df.shape}")
df.head()

## 4. Forbered data

In [ ]:
print("Kolonner:", list(df.columns))
print("\nManglende værdier:")
print(df.isna().sum())
df.describe()

In [ ]:
feature_cols = [
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "temperature",
    "is_start_of_semester",
    "is_during_semester",
    "month",
    "hour",
]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"Mangler kolonner i CSV: {missing}")

X = df[feature_cols].copy()

def to_occupancy_class(n):
    if n <= 20:
        return 0
    if n <= 50:
        return 1
    return 2

y_class = df["number_people"].apply(to_occupancy_class)

class_names = ["Roligt", "Moderat", "Travlt"]
print("Klassefordeling:")
print(y_class.value_counts().sort_index().rename(index=dict(enumerate(class_names))))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=y_class.map(dict(enumerate(class_names))), order=class_names)
plt.title("Fordeling af occupancy-klasser")
plt.xlabel("Klasse")
plt.ylabel("Antal målinger")
plt.tight_layout()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_class,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_class,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_oh = to_categorical(y_train, num_classes=3)
y_test_oh = to_categorical(y_test, num_classes=3)

print("X_train:", X_train_scaled.shape)
print("y_train one-hot:", y_train_oh.shape)
print("Eksempel label:", y_train_oh[0])

## 5. Træn DNN-model

Jeg bygger et Sequential-netværk med Dense-lag, som i pensum.

In [ ]:
n_features = X_train_scaled.shape[1]

model = Sequential()
model.add(Dense(32, input_dim=n_features, activation="relu"))
model.add(Dense(16, activation="relu"))
model.add(Dense(3, activation="softmax"))

adam = optimizers.Adam(learning_rate=0.001)
model.compile(
    loss="categorical_crossentropy",
    optimizer=adam,
    metrics=["accuracy"],
)

model.summary()

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train_oh,
    validation_split=0.2,
    epochs=40,
    batch_size=64,
    verbose=1,
)

## 6. Evaluér modellens kvalitet

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
y_prob = model.predict(X_test_scaled)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.asarray(y_test)

acc = accuracy_score(y_true, y_pred)
print(f"Test accuracy: {acc:.3f}\n")
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

## 7. Manuel prediction

Jeg tester én ny observation med `model.predict`.

In [ ]:
sample = pd.DataFrame(
    [{
        "day_of_week": 1,
        "is_weekend": 0,
        "is_holiday": 0,
        "temperature": 64.4,
        "is_start_of_semester": 0,
        "is_during_semester": 1,
        "month": 3,
        "hour": 17,
    }]
)

sample_scaled = scaler.transform(sample)
prediction = model.predict(sample_scaled)
pred_class = int(np.argmax(prediction[0]))

print("Raw output (softmax):", prediction)
print(f"Predicted class: {pred_class} = {class_names[pred_class]}")
for i, name in enumerate(class_names):
    print(f"  {name}: {prediction[0][i]:.3f}")

## 8. Konklusion

Jeg har klassificeret travlhed i et fitnesscenter med en DNN.

- **Data:** Kaggle campus gym crowdedness
- **Prep:** Feature-valg, skalering, 3 klasser, one-hot, train/test
- **Model:** DNN (`Dense` + ReLU + Softmax + Adam)
- **Evaluering:** Accuracy, confusion matrix, classification report, loss-kurve